# Dt 谱 × meta-gradient 金标准
**回答的问题：Dt 作为"下游贡献度代理"有没有价值。**

meta-grad（一阶）：∂L_task/∂w(t) ∝ −⟨∇L_task, ∇L_diff(t)⟩，是该 t 对下游任务真实增益的直接度量。Dt 谱与 meta-grad 谱的 Spearman/Pearson 相关性越高，Dt 越可信。

支持对多个 checkpoint 批量测量并汇总相关性演化。

**v2：Dt 候选定义裁判 + 可靠性升级**

动机：绝对 loss 差没有消除跨 t 的量纲差异（大 t 处 loss 天然大一个数量级），D_abs 做采样权重等于让量纲因素在梯度强度和采样频率上乘两次。因此并测三个候选，谁与 meta-grad 相关高且跨 ckpt 稳定谁胜出：
- **dabs**：mean(L0)−mean(Lz)，原始绝对差（对照）；
- **drel**：1 − mean(Lz)/mean(L0)，相对消除率（ratio-of-means，尺度无关，"贡献率"语义）；
- **deff**：(mean(L0)−mean(Lz))/std(L0)，效应量（量纲与可靠性合并）；
- **dx0**：median‖p_z−p_c‖²，线上控制器 d_table 的实际口径（对照）。
（候选③"D/EMA(L0)"与 drel 逐点相等，不单列；loss 固定 x0 预测，ε 空间口径已移除。）

可靠性设计：两轴数据劈半（防共享数据虚增相关）、双侧各 8 噪声 CRN 重复（每点 128 样本）、相关系数带桶级 bootstrap 95% CI。

meta 轴归一化：裸内积 ⟨g_task, g_diff⟩ 混有去噪梯度量级随 t 增长的混杂（大 t 处 loss 大、梯度大），
主量改为单位化内积 ⟨g_task, g_diff/‖g_diff‖⟩——"沿该 t 方向走单位步长的 task loss 变化"，跨 t 可比；
裸内积保留在 mg['meta_raw'] 作诊断，cos（mg['meta_cos']）与单位化内积跨 t 排名相同。

meta 轴全网格化：不再只测 16 个桶代表 t，而是和 Dt 一样在训练网格每个 t 上测量（T_STRIDE 可调），
相关直接在 ~100 对网格点上计算（旧版只有 16 桶点），曲线连续、CI 更紧。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把两个路径都算成绝对路径 ----
VISUAL = os.path.abspath('.')                                  # 内核启动时 cwd 就是 visual/
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import probe_core as pc

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CFG 用 run 目录里 train 时 dump 的完整 config，口径与训练严格一致；CKPTS 按 iter 排序。
# 可靠性设计（v2）：
#   - 两个轴用不同数据：32 张图劈半，Dt 谱用前半、meta-grad 用后半——
#     共享数据会共同原因虚增相关，劈半是防线；
#   - 两个轴各自 CRN 多噪声重复（N_NOISE=8，各自独立种子），
#     每 t/桶有效样本 8×16=128，曲线假抖动与均值噪声同步压低；
#   - 相关系数带桶级 bootstrap 95% CI（plot_dt_vs_meta 内自动算）。
import glob, re
RUN_DIR = 'work_dirs/eden/fcn_diffae'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]   # run 目录里 dump 的 config
CKPTS = sorted(glob.glob(f'{RUN_DIR}/iter_*.pth'),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = 'work_dirs/offline_probe/dt_meta'
NUM_IMAGES = 32       # 劈半：前 16 给 Dt 谱，后 16 给 meta-grad
N_NOISE = 8           # 每轴噪声重复数（CRN，各自独立种子）
T_STRIDE = 1          # meta-grad 全网格测量：1=每个 t 都测（800 次反传/ckpt），2=隔点（减半）
SEED = 0
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
rows = []
# 数据劈半：Dt 与 meta-grad 用不同图像（防共享数据虚增相关）
half = batch['x_diffae'].shape[0] // 2
def _sub(b, sl):
    return {'cond': b['cond'][sl], 'x_diffae': b['x_diffae'][sl],
            'xis': b['xis'][sl], 'dsis': b['dsis'][sl]}
batch_dt = _sub(batch, slice(0, half))
batch_mg = _sub(batch, slice(half, 2 * half))
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    dt = pc.measure_dt_spectrum(model, batch_dt, n_noise=N_NOISE, seed=SEED)
    mg = pc.measure_meta_grad_spectrum(model, batch_mg, n_noise=N_NOISE, seed=SEED + 1, t_stride=T_STRIDE)
    fig, corrs = pc.plot_dt_vs_meta(
        dt, mg, title=tag,
        save_path=os.path.join(OUT, f"{tag}_dt_vs_meta.png"))
    rows.append((tag, corrs))
    s = ' | '.join(f"{k}: ρ={v[0]:.3f}{'CI[%0.2f,%0.2f]' % v[3] if v[3][0] == v[3][0] else ''}"
                   for k, v in corrs.items())
    print(f"{tag}: {s}")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 汇总：三个 Dt 变体与 meta-grad 的相关性随训练演化
import matplotlib.pyplot as plt
if len(rows) > 1:
    tags = [r[0] for r in rows]
    keys = list(rows[0][1].keys())
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for k in keys:
        axes[0].plot(tags, [r[1][k][0] for r in rows], 'o-', label=k)
        axes[1].plot(tags, [r[1][k][2] for r in rows], 's-', label=k)
    for ax, name in zip(axes, ['Spearman ρ', 'Pearson r']):
        ax.axhline(0, color='k', ls='--', lw=0.8)
        ax.set_ylim(-1, 1); ax.grid(alpha=0.3); ax.legend(fontsize=8)
        ax.set_title(f'Dt ~ meta-grad {name} across checkpoints')
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.savefig(os.path.join(OUT, 'summary.png'), dpi=150, bbox_inches='tight')
    fig